# FLAG 2027 - Notebook 40 RUN-ALL: fine-tune the ECAPA voice encoder cross-modally (the honest leaders' lever)

Every entry above 21 on the board either processes the test set as a whole (banned for the evaluation phase) or trained its encoders cross-modally on a large VoxCeleb slice (FAME 2024 winner, 19.9: fine-tuned ECAPA and ResNet; removing that stage cost them 6 points). We have only ever trained a head on frozen features, and thirteen notebooks say the frozen-feature ceiling of our stack is about 22. Our one fine-tuning pilot (NB16, LoRA on WavLM, 184 identities) came out worse; the data was too small, not the idea.

This notebook fine-tunes **ECAPA-TDNN** (speechbrain `yangwang825/ecapa-tdnn-vox2`, 192-d, the smallest and fastest of our six voice encoders and the one the 2024 winner fine-tuned) with:

- data: VoxCeleb2 parts `vox2_test_mp4` + `vox2_mp4_1` (re-downloaded, the SAME rows as our stored features, Bengali removed by language-ID as always) + MAV-Celeb v1/v3/extra. About 3,300 identities, 76,000 clips. **Challenge train audio is never used for fine-tuning**, so the CV test afterwards is clean and the challenge rows serve as an unseen report.
- loss: SupCon L_out (the form that reached the board) between a projection of the ECAPA embedding and a projection of the frozen CLIP face embedding, plus an AAM-softmax speaker loss whose class weights start at the frozen per-identity means, so the encoder keeps its speaker discrimination while it learns face-relevant cues.
- recipe: 16 identities x 4 clips per step, random 3-s crops of a 6-s centre window, encoder learning rate 1e-5, fp16; 300 warm-up steps with the encoder frozen; up to 6,000 steps; checkpoint selection on 10% held-out external identities (never on challenge rows); checkpoints on Drive, so a disconnect loses at most 500 steps of training.
- then: the fine-tuned ECAPA is extracted for all training rows (challenge, MAV, both VoxCeleb parts) and tested in system E's recipe on the usual CV (stage A + B), added next to the frozen ECAPA (`V_eft_add`) and replacing it (`V_eft_rep`). Same rule as always.

Rules: an external audio encoder fine-tuned on VoxCeleb (no Bengali) and MAV-Celeb (English, Urdu, Hindi, German) is allowed and will be declared; no test data is involved anywhere; scoring stays per-trial.

**How to run:** T4 GPU, `HF_TOKEN` in Colab Secrets, Runtime > Run all. About 4.5-5 h (download ~40 min, ffmpeg + language-ID ~35 min, waveform cache ~20 min, fine-tuning ~1.5-2 h, extraction ~40 min, CV ~25 min). Overnight. If the session dies, Run all again: data is rebuilt (~1.5 h), training resumes from the Drive checkpoint, finished extraction and CV runs are skipped. Paste the **SUMMARY**.


In [ ]:
# Cell 1 [NB40 v1 5-Nov] - Drive, GPU (hard check), packages, HF token (Colab Secrets only; never paste it anywhere)
from google.colab import drive, userdata
drive.mount('/content/drive')
!pip -q install speechbrain huggingface_hub
import torch, shutil
assert torch.cuda.is_available(), 'NO GPU in this runtime. Runtime > Change runtime type > T4 GPU > Save, then run again.'
dev = 'cuda'
HF_TOKEN = userdata.get('HF_TOKEN')
assert HF_TOKEN, 'HF_TOKEN not found in Colab Secrets (key icon on the left): add it and enable notebook access, then run again.'
print('[NB40 v1 5-Nov] Device:', torch.cuda.get_device_name(0), '| HF token found: True | disk free %.0f GB' % (shutil.disk_usage('/content')[2] / 1e9))


In [ ]:
# Cell 2 [NB40 v1 5-Nov] - audio + face-CLIP tables in our row order: challenge train (report only), MAV v1/v3 + extra, VoxCeleb2 vox2_test_mp4 + vox2_mp4_1
#          (re-downloaded, SAME rows as the stored features: NB07 selection rule, Bengali removed, ECAPA alignment check) -> 6-s waveform cache on local disk
import os, re, json, time, zipfile, subprocess, glob, copy
import numpy as np, pandas as pd, librosa, soundfile as sf
import torch.nn as nn, torch.nn.functional as Fn
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor
from huggingface_hub import hf_hub_download
BASE = '/content/drive/MyDrive'
PROJECT = os.path.join(BASE, 'FLAG2027'); FEATDIR = os.path.join(PROJECT, 'features'); RESDIR = os.path.join(PROJECT, 'results'); CACHE = os.path.join(PROJECT, 'models_cache')
FINAL = os.path.join(PROJECT, 'final'); os.makedirs(FINAL, exist_ok=True)
ZIP_DIR = json.load(open(os.path.join(PROJECT, 'paths.json')))['ZIP_DIR']
MAV_ZIPS, MAV_LISTS = os.path.join(BASE, 'dataset'), os.path.join(BASE, 'face_voice_association_splits')
SUMMARY = []
def note(msg):
    print(msg); SUMMARY.append(msg)
def l2(X):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
try:
    from speechbrain.inference.interfaces import Pretrained
    from speechbrain.inference.classifiers import EncoderClassifier
except ImportError:
    from speechbrain.pretrained.interfaces import Pretrained
    from speechbrain.pretrained import EncoderClassifier
class Encoder(Pretrained):
    MODULES_NEEDED = ['compute_features', 'mean_var_norm', 'embedding_model']
    def encode_batch(self, wavs, wav_lens=None):
        if wav_lens is None:
            wav_lens = torch.ones(wavs.shape[0], device=self.device)
        wavs, wav_lens = wavs.to(self.device).float(), wav_lens.to(self.device)
        return self.mods.embedding_model(self.mods.mean_var_norm(self.mods.compute_features(wavs), wav_lens), wav_lens)
ECAPA = Encoder.from_hparams(source='yangwang825/ecapa-tdnn-vox2', savedir=os.path.join(CACHE, 'ecapa_vox2'), run_opts={'device': dev}); ECAPA.eval()
def batches_by_length(paths, bs):
    order = np.argsort([sf.info(p).duration for p in paths])
    for i in range(0, len(order), bs):
        yield order[i:i + bs]
def pad(paths, idx, max_sec=None):
    wavs = [librosa.load(paths[j], sr=16000, mono=True)[0] for j in idx]
    if max_sec:
        wavs = [w[:16000 * max_sec] for w in wavs]
    lens = torch.tensor([len(w) for w in wavs], dtype=torch.float); L = int(lens.max()); b = torch.zeros(len(wavs), L)
    for k, w in enumerate(wavs):
        b[k, :len(w)] = torch.from_numpy(w)
    return b, lens / L
def enc_ecapa_paths(paths, bs=16, desc='ecapa'):
    """full-clip ECAPA with whatever weights ECAPA.mods.embedding_model currently holds (identical batching to NB20 Cell 4)"""
    ECAPA.mods.embedding_model.eval(); out = np.zeros((len(paths), 192), np.float32)
    for idx in tqdm(list(batches_by_length(paths, bs)), desc=desc):
        b, rl = pad(paths, idx)
        with torch.no_grad():
            out[idx] = ECAPA.encode_batch(b, rl).squeeze(1).float().cpu().numpy()
    return out
# ---- challenge train (never fine-tuned on; used only to report EER during training and for the CV afterwards) ----
FULL = '/content/flag_data'; os.makedirs(FULL, exist_ok=True)
if not os.path.exists(os.path.join(FULL, 'train_set.zip.done')):
    with zipfile.ZipFile(os.path.join(ZIP_DIR, 'train_set.zip')) as zf:
        zf.extractall(FULL, members=[m for m in zf.namelist() if m.lower().endswith(('.wav', '.txt', '.csv'))])
    open(os.path.join(FULL, 'train_set.zip.done'), 'w').close()
TXT = [os.path.join(r, f) for r, _, fs in os.walk(FULL) for f in fs if f.endswith(('.txt', '.csv'))]
def find_one(*parts):
    hits = [p for p in TXT if all(x in p for x in parts)]
    assert len(hits) == 1, 'looking for %s -> found %s' % (parts, hits)
    return hits[0]
def resolve(rel, list_path):
    q = os.path.dirname(list_path)
    while q.startswith(FULL):
        c = os.path.join(q, rel)
        if os.path.exists(c):
            return c
        q = os.path.dirname(q)
def tok(tokens, exts, fallback):
    t = [x for x in tokens if x.lower().endswith(exts)]; return t[0] if len(t) == 1 else tokens[fallback]
def read_feat(path, dim):
    df = pd.read_csv(path, header=None)
    return df.iloc[:, :dim].values.astype(np.float32), (df.iloc[:, dim].values if df.shape[1] > dim else None)
lp = find_one('train_English.txt'); lines = [l.split() for l in open(lp) if l.strip()]
WAV_CH = [resolve(tok(t, ('.wav',), 2), lp) for t in lines]; assert None not in WAV_CH, 'challenge wav files not found - tell Claude'
Xv, y = read_feat(find_one('train_English_voices.csv'), 192); y = y.astype(int); assert len(y) == len(WAV_CH)
cls2spk = {int(l[5]): l[4] for l in lines}
meta = pd.read_csv(os.path.join(ZIP_DIR, 'meta_file_train_set.csv'), header=None, names=['spk', 'g'])
spk2g = dict(zip(meta.spk.astype(str).str.strip(), meta.g.astype(str).str.strip().str.lower()))
ids = np.unique(y); id_gender = {c: spk2g[cls2spk[c]] for c in ids}
CLIP_CH = np.load(os.path.join(FEATDIR, 'clip.npz'))['train']; assert len(CLIP_CH) == len(y)
# ---- MAV v1/v3 voices (same rows as mavceleb_v1v3.npz) ----
def zip_index(zf):
    m = {}
    for n in zf.namelist():
        for key in ('voices/', 'faces/'):
            i = n.find(key)
            if i >= 0 and not n.endswith('/'):
                m[n[i:]] = n; break
    return m
MAVD = '/content/mav'; mav_v, mspk = [], []
for ver, zname, lists in [('v1', 'mavceleb_v1_train.zip', ['English_train.txt', 'Urdu_train.txt']), ('v3', 'mavceleb_v3_tran.zip', ['English_train.txt', 'German_train.txt'])]:
    flag = os.path.join(MAVD, ver + '.voices.done'); need = not os.path.exists(flag)
    zf = zipfile.ZipFile(os.path.join(MAV_ZIPS, zname)); zi = zip_index(zf)
    for lf in lists:
        for l in open(os.path.join(MAV_LISTS, ver, lf)):
            t = l.split()
            if len(t) < 2:
                continue
            vm, fm = zi.get('voices/' + t[0]), zi.get('faces/' + t[1])
            if vm is None or fm is None:
                continue
            if need:
                zf.extract(vm, os.path.join(MAVD, ver))
            mav_v.append(os.path.join(MAVD, ver, vm)); mspk.append(ver + '_' + t[0].split('/')[0])
    zf.close()
    if need:
        open(flag, 'w').close()
MAV = dict(np.load(os.path.join(FEATDIR, 'mavceleb_v1v3.npz'))); assert list(MAV['spk']) == mspk, 'MAV v1/v3 row order differs - tell Claude'
# ---- MAV v2 + v1-complete 'extra' voices (same rng(0) sequence as NB24/NB35/NB39, so the rows line up with mavceleb_extra.npz) ----
EXTRA = dict(np.load(os.path.join(FEATDIR, 'mavceleb_extra.npz'))); MAX_PER_ID = 120
VPAT = re.compile(r'voices/(id\d+)/([^/]+)/([^/]+)/([^/]+\.wav)$'); FPAT = re.compile(r'faces/(id\d+)/([^/]+)/([^/]+)/([^/]+\.jpg)$')
v1_used = set()
for lf in ['English_train.txt', 'Urdu_train.txt']:
    for l in open(os.path.join(MAV_LISTS, 'v1', lf)):
        t = l.split()
        if t:
            v1_used.add(t[0].split('/')[0])
rows = []
for ver, zname, skip_ids in [('v2', 'mavceleb_v2_complete.zip', set()), ('v1', 'mavceleb_v1_complete.zip', v1_used)]:
    D = '/content/mavx/' + ver; flag = os.path.join(D, 'voices.done'); need = not os.path.exists(flag)
    zf = zipfile.ZipFile(os.path.join(MAV_ZIPS, zname)); voices, faces = {}, {}
    for n in zf.namelist():
        mv = VPAT.search(n)
        if mv:
            voices.setdefault(mv.group(1), []).append((mv.group(2), mv.group(3), n)); continue
        mf = FPAT.search(n)
        if mf:
            faces.setdefault((mf.group(1), mf.group(2), mf.group(3)), []).append(n)
    ids_ = sorted(i for i in voices if i not in skip_ids); rng = np.random.default_rng(0)
    for pid in ids_:
        vs = voices[pid]; vs = [vs[j] for j in rng.permutation(len(vs))[:MAX_PER_ID]]
        for lang_, vid, vn in vs:
            fr = faces.get((pid, lang_, vid)) or [f for (p, l, v), fl in faces.items() if p == pid for f in fl]
            if not fr:
                continue
            _ = fr[rng.integers(len(fr))]                 # keeps the random sequence identical to the face extraction
            if need:
                zf.extract(vn, D)
            rows.append((ver + '_' + pid, os.path.join(D, vn)))
    zf.close()
    if need:
        open(flag, 'w').close()
spk_all = np.array([r[0] for r in rows]); keep = np.ones(len(rows), bool)
if len(rows) != len(EXTRA['spk']):
    j = 0; keep[:] = False
    for i in range(len(rows)):
        if j < len(EXTRA['spk']) and spk_all[i] == EXTRA['spk'][j]:
            keep[i] = True; j += 1
    assert j == len(EXTRA['spk']), 'could not align MAV extra rows - tell Claude'
extra_v = [r[1] for r, k in zip(rows, keep) if k]
assert [r[0] for r, k in zip(rows, keep) if k] == list(EXTRA['spk']), 'MAV extra row order differs - tell Claude'
# ---- VoxCeleb2: the two parts system E trains on, rebuilt exactly (NB26 Cell 3 logic), audio only ----
LID = EncoderClassifier.from_hparams(source='speechbrain/lang-id-voxlingua107-ecapa', savedir=os.path.join(CACHE, 'lid_voxlingua107'), run_opts={'device': dev})
def language_of(paths, bs=32):
    out = [''] * len(paths)
    for idx in tqdm(list(batches_by_length(paths, bs)), desc='language-ID'):
        b, rl = pad(paths, idx, max_sec=10)
        with torch.no_grad():
            labs = LID.classify_batch(b.to(dev), rl.to(dev))[3]
        for j, lab in zip(idx, labs):
            out[j] = lab.split(':')[0].strip()
    return np.array(out)
def ffmpeg_wav(job):
    mp4, wav = job
    a = subprocess.run(['ffmpeg', '-loglevel', 'error', '-y', '-i', mp4, '-vn', '-ac', '1', '-ar', '16000', wav]).returncode
    return a == 0 and os.path.exists(wav)
ZIPS = [('vox2/vox2_test_mp4.zip', 40), ('vox2/vox2_mp4_1.zip', 20)]
VOX = {}
for zname, cpi in ZIPS:
    tag = os.path.basename(zname).replace('.zip', ''); OLD = dict(np.load(os.path.join(FEATDIR, 'vox_' + tag + '.npz')))
    W = '/content/vox_work/' + tag; os.makedirs(W, exist_ok=True); rows_local = os.path.join(W, 'rows.npz'); t0 = time.time()
    if os.path.exists(rows_local):
        r = np.load(rows_local); wavs = [os.path.join(W, x) for x in r['wav']]; spk = r['spk']
    else:
        zpath = hf_hub_download('ProgramComputer/voxceleb', zname, repo_type='dataset', token=HF_TOKEN, local_dir='/content/vox')
        zf = zipfile.ZipFile(zpath); tree = {}
        for n in zf.namelist():
            mm = re.search(r'(id\d{5})/([^/]+)/([^/]+\.mp4)$', n)
            if mm:
                tree.setdefault(mm.group(1), {}).setdefault(mm.group(2), []).append(n)
        rng = np.random.default_rng(0); picked = []
        for pid, vids in tree.items():
            pools = [list(rng.permutation(v)) for v in vids.values()]; chosen = []
            while len(chosen) < cpi and any(pools):
                for p in pools:
                    if p and len(chosen) < cpi:
                        chosen.append(p.pop())
            picked += [(pid, c) for c in chosen]
        jobs = []
        for i, (pid, member) in enumerate(picked):
            jobs.append((zf.extract(member, W + '/mp4'), '%s/%06d.wav' % (W, i)))
        zf.close(); os.remove(zpath)
        with ThreadPoolExecutor(8) as ex:
            ok = list(tqdm(ex.map(ffmpeg_wav, jobs), total=len(jobs), desc='ffmpeg ' + tag))
        shutil.rmtree(W + '/mp4', ignore_errors=True)
        keep_ = [i for i, g in enumerate(ok) if g]
        wavs = [jobs[i][1] for i in keep_]; spk = np.array(['vox_' + picked[i][0] for i in keep_])
        lang = language_of(wavs); nb = lang != 'bn'
        wavs = [w for w, k in zip(wavs, nb) if k]; spk = spk[nb]
        np.savez(rows_local, wav=np.array([os.path.basename(w) for w in wavs]), spk=spk)
    assert len(spk) == len(OLD['spk']) and (spk == OLD['spk']).all(), '%s: row order differs from the stored features - tell Claude' % tag
    probe = np.random.default_rng(1).choice(len(wavs), 300, replace=False)
    e_new = enc_ecapa_paths([wavs[j] for j in probe], desc='ecapa check ' + tag); e_old = OLD['ecapa'][probe]
    cos = (e_new * e_old).sum(1) / (np.linalg.norm(e_new, axis=1) * np.linalg.norm(e_old, axis=1) + 1e-9)
    note('%s: %d rows, %d identities | alignment median ECAPA cosine %.4f | %.0f min' % (tag, len(spk), len(set(spk)), np.median(cos), (time.time() - t0) / 60))
    assert np.median(cos) > 0.98, '%s: rows do not line up - stop and tell Claude' % tag
    VOX[tag] = dict(wav=wavs, spk=list(spk), clip=OLD['clip'], ecapa=OLD['ecapa'])
del LID; torch.cuda.empty_cache()
# ---- master tables: rows = challenge | mav | extra | vox_test | vox_1 ----
TAGS = [os.path.basename(z).replace('.zip', '') for z, _ in ZIPS]
ALL_WAV = WAV_CH + mav_v + extra_v + sum([VOX[t]['wav'] for t in TAGS], [])
ALL_SPK = np.array(['ch_%d' % c for c in y] + list(MAV['spk']) + list(EXTRA['spk']) + sum([VOX[t]['spk'] for t in TAGS], []))
ALL_CLIP = np.concatenate([CLIP_CH, MAV['clip'], EXTRA['clip']] + [VOX[t]['clip'] for t in TAGS]).astype(np.float32)
ALL_ECAPA0 = np.concatenate([Xv, MAV['ecapa'], EXTRA['ecapa']] + [VOX[t]['ecapa'] for t in TAGS]).astype(np.float32)
N_CH, N_MAV, N_EX = len(y), len(MAV['spk']), len(EXTRA['spk'])
SEG = np.r_[np.zeros(N_CH, int), np.ones(N_MAV, int), 2 * np.ones(N_EX, int)]
for i, t in enumerate(TAGS):
    SEG = np.r_[SEG, (3 + i) * np.ones(len(VOX[t]['spk']), int)]
assert len(ALL_WAV) == len(ALL_SPK) == len(ALL_CLIP) == len(ALL_ECAPA0) == len(SEG)
note('rows: challenge %d | MAV v1/v3 %d | MAV extra %d | %s | total %d clips, %d identities' % (N_CH, N_MAV, N_EX, ' | '.join('%s %d' % (t, len(VOX[t]['spk'])) for t in TAGS), len(ALL_SPK), len(set(ALL_SPK))))
# ---- 6-s int16 waveform cache (centre window; short clips tiled), random 3-s crops are taken from it during training ----
SEC = 6; L6 = 16000 * SEC
def load6(p):
    w = librosa.load(p, sr=16000, mono=True)[0]
    if len(w) >= L6:
        s0 = (len(w) - L6) // 2; w = w[s0:s0 + L6]
    else:
        w = np.tile(w, int(np.ceil(L6 / max(len(w), 1))))[:L6]
    return (np.clip(w, -1, 1) * 32767).astype(np.int16)
CACHE6 = '/content/wav6.npy'
if not os.path.exists(CACHE6 + '.done'):
    t0 = time.time(); arr = np.lib.format.open_memmap(CACHE6, mode='w+', dtype=np.int16, shape=(len(ALL_WAV), L6))
    with ThreadPoolExecutor(8) as ex:
        for i, w in enumerate(tqdm(ex.map(load6, ALL_WAV), total=len(ALL_WAV), desc='decode 6-s waveforms')):
            arr[i] = w
    arr.flush(); del arr; open(CACHE6 + '.done', 'w').close()
    note('waveform cache built: %d x %d s in %.0f min (%.1f GB)' % (len(ALL_WAV), SEC, (time.time() - t0) / 60, os.path.getsize(CACHE6) / 1e9))
WAVE = np.load(CACHE6, mmap_mode='r')


In [ ]:
# Cell 3 [NB40 v1 5-Nov] - fine-tune ECAPA cross-modally: SupCon L_out between proj(ECAPA) and proj(frozen CLIP face) + AAM speaker loss
#          (class weights start at the frozen per-identity means). Data = external rows only (MAV + VoxCeleb); 10% of external identities held out
#          for checkpoint selection; challenge rows are reported but never trained on or selected on. Checkpoints on Drive (resumable).
from sklearn.metrics import roc_curve
EMB = ECAPA.mods.embedding_model
for p in EMB.parameters():
    p.requires_grad = True
ECAPA.mods.compute_features.eval(); ECAPA.mods.mean_var_norm.eval()
EXT_ROWS = np.where(SEG > 0)[0]
ext_ids = sorted(set(ALL_SPK[EXT_ROWS])); rng0 = np.random.default_rng(0); perm = rng0.permutation(len(ext_ids)); n_val = int(round(0.1 * len(ext_ids)))
VAL_IDS = set(ext_ids[i] for i in perm[:n_val]); TR_IDS = sorted(ext_ids[i] for i in perm[n_val:])
tr_rows = np.array([i for i in EXT_ROWS if ALL_SPK[i] not in VAL_IDS]); val_rows = np.array([i for i in EXT_ROWS if ALL_SPK[i] in VAL_IDS])
cls_of = {s: i for i, s in enumerate(TR_IDS)}; ytr = np.array([cls_of[ALL_SPK[i]] for i in tr_rows]); pos_by_cls = [np.where(ytr == c)[0] for c in range(len(TR_IDS))]
note('fine-tuning identities %d (%d clips) | held-out external identities %d (%d clips) | challenge rows %d: report only, never fine-tuned on' % (len(TR_IDS), len(tr_rows), len(VAL_IDS), len(val_rows), N_CH))
class Proj(nn.Module):
    def __init__(self, d_in, d=128):
        super().__init__(); self.net = nn.Sequential(nn.Linear(d_in, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Linear(256, d))
    def forward(self, x):
        return self.net(x)
proj_v, proj_f = Proj(192).to(dev), Proj(768).to(dev)
Wc = np.stack([l2(ALL_ECAPA0[tr_rows[pos]]).mean(0) for pos in pos_by_cls]); W_SPK = nn.Parameter(torch.tensor(l2(Wc), device=dev, dtype=torch.float32))
def aam_loss(e, lab, m=0.2, s=30.0):
    cos = Fn.normalize(e, dim=1) @ Fn.normalize(W_SPK, dim=1).t(); oh = Fn.one_hot(lab, cos.shape[1]).bool()
    theta = torch.acos(cos.clamp(-1 + 1e-6, 1 - 1e-6)); logits = s * torch.where(oh, torch.cos(theta + m), cos)
    return Fn.cross_entropy(logits, lab)
def lout(a, b, lab, tau=0.07):
    S = Fn.normalize(a, dim=1) @ Fn.normalize(b, dim=1).t() / tau; P = (lab[:, None] == lab[None, :]).float()
    def one(M):
        return -(P * Fn.log_softmax(M, 1)).sum(1).div(P.sum(1)).mean()
    return 0.5 * (one(S) + one(S.t()))
CLIP_T = torch.tensor(l2(ALL_CLIP), device=dev)
def ecapa_feats(wav):
    with torch.no_grad():
        lens = torch.ones(wav.shape[0], device=dev); f = ECAPA.mods.mean_var_norm(ECAPA.mods.compute_features(wav), lens)
    return f, lens
USE_HALF = [True]                                     # fp16 autocast for the encoder; switched off automatically if losses turn non-finite
def embed_train(wav):
    f, lens = ecapa_feats(wav)
    with torch.autocast('cuda', dtype=torch.float16, enabled=USE_HALF[0]):
        e = EMB(f, lens)
    return e.squeeze(1).float()
@torch.no_grad()
def embed_rows(rows, bs=128):
    EMB.eval(); out = []
    for i in range(0, len(rows), bs):
        r = rows[i:i + bs]; wav = torch.tensor(WAVE[r].astype(np.float32) / 32767, device=dev); f, lens = ecapa_feats(wav)
        with torch.autocast('cuda', dtype=torch.float16, enabled=USE_HALF[0]):
            e = EMB(f, lens)
        out.append(e.squeeze(1).float())
    return torch.cat(out)
def eer(labels, scores):
    fpr, tpr, _ = roc_curve(labels, scores); fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr)); return (fpr[i] + fnr[i]) / 2 * 100
def pairs_of(rows, n, seed, gender=None):
    rng_ = np.random.default_rng(seed); spk_ = np.array(sorted(set(ALL_SPK[rows]))); by = {s: rows[ALL_SPK[rows] == s] for s in spk_}
    V, F, Lb = [], [], []
    for _ in range(n):
        s = rng_.choice(spk_); V.append(rng_.choice(by[s])); F.append(rng_.choice(by[s])); Lb.append(1)
        s = rng_.choice(spk_)
        cand = [t for t in spk_ if t != s and (gender is None or gender[t] == gender[s])] or [t for t in spk_ if t != s]
        t = rng_.choice(cand); V.append(rng_.choice(by[s])); F.append(rng_.choice(by[t])); Lb.append(0)
    return np.array(V), np.array(F), np.array(Lb)
CH_ROWS = np.arange(N_CH); ch_gender = {'ch_%d' % c: id_gender[c] for c in ids}
PAIRS = {'val_ext': pairs_of(val_rows, 3000, 0), 'ch_std': pairs_of(CH_ROWS, 2000, 0), 'ch_gender': pairs_of(CH_ROWS, 2000, 100, ch_gender)}
@torch.no_grad()
def evaluate():
    proj_v.eval(); proj_f.eval(); res = {}
    for name, (Vi, Fi, Lb) in PAIRS.items():
        rows = np.unique(np.r_[Vi, Fi]); pos = {int(r): i for i, r in enumerate(rows)}
        e = embed_rows(rows); pv = Fn.normalize(proj_v(e), dim=1); pf = Fn.normalize(proj_f(CLIP_T[torch.tensor(rows, device=dev)]), dim=1)
        iv = torch.tensor([pos[int(r)] for r in Vi], device=dev); i_f = torch.tensor([pos[int(r)] for r in Fi], device=dev)
        res[name] = eer(Lb, (pv[iv] * pf[i_f]).sum(1).cpu().numpy())
    proj_v.train(); proj_f.train(); return res
def fmt(r):
    return 'held-out external EER %.2f | challenge std %.2f gender %.2f' % (r['val_ext'], r['ch_std'], r['ch_gender'])
P_ID, K_CLIP, CROP = 16, 4, 16000 * 3
TOTAL, WARMUP, EVAL_EVERY, PATIENCE, LAM_SPK = 6000, 300, 500, 4, 1.0
opt = torch.optim.AdamW([{'params': list(EMB.parameters()), 'lr': 1e-5, 'weight_decay': 0.0},
                         {'params': list(proj_v.parameters()) + list(proj_f.parameters()), 'lr': 1e-3, 'weight_decay': 0.01},
                         {'params': [W_SPK], 'lr': 1e-4, 'weight_decay': 0.0}])
scaler = torch.amp.GradScaler('cuda')
CKPT, BESTP = os.path.join(FINAL, 'ecapa_ft_ckpt.pt'), os.path.join(FINAL, 'ecapa_ft.pt')
CFG = dict(encoder='yangwang825/ecapa-tdnn-vox2', data='MAV v1/v3/extra + VoxCeleb2 vox2_test_mp4 + vox2_mp4_1 (bn removed), challenge rows excluded', loss='SupCon L_out tau 0.07 vs frozen CLIP projection + AAM(m 0.2, s 30) speaker loss', batch='16 ids x 4 clips, 3-s crops', lr='encoder 1e-5, projections 1e-3, class weights 1e-4', warmup=WARMUP, total=TOTAL)
step, best, best_step, wait, hist, ref = 0, 1e9, 0, 0, [], None
rng = np.random.default_rng(1); torch.manual_seed(1)
if os.path.exists(CKPT):
    ck = torch.load(CKPT, map_location=dev, weights_only=False)
    EMB.load_state_dict(ck['emb']); proj_v.load_state_dict(ck['proj_v']); proj_f.load_state_dict(ck['proj_f']); W_SPK.data.copy_(ck['w_spk']); opt.load_state_dict(ck['opt'])
    step, best, best_step, wait, hist, ref = ck['step'], ck['best'], ck['best_step'], ck['wait'], ck['hist'], ck['ref']; rng = np.random.default_rng(1 + step)
    print('resumed from step %d (best held-out EER %.2f at step %d)' % (step, best, best_step))
def save_ckpt():
    torch.save({'emb': EMB.state_dict(), 'proj_v': proj_v.state_dict(), 'proj_f': proj_f.state_dict(), 'w_spk': W_SPK.data, 'opt': opt.state_dict(), 'step': step, 'best': best, 'best_step': best_step, 'wait': wait, 'hist': hist, 'ref': ref, 'cfg': CFG}, CKPT)
proj_v.train(); proj_f.train(); t0 = time.time(); n_done = 0; n_bad = 0
while step < TOTAL and wait < PATIENCE:
    chosen = rng.choice(len(TR_IDS), P_ID, replace=False)
    j = np.concatenate([rng.choice(pos_by_cls[c], K_CLIP, replace=len(pos_by_cls[c]) < K_CLIP) for c in chosen]); rws = tr_rows[j]
    s0 = rng.integers(0, L6 - CROP + 1, len(rws))
    wav = torch.tensor(np.stack([WAVE[r, s:s + CROP] for r, s in zip(rws, s0)]).astype(np.float32) / 32767, device=dev); lab = torch.tensor(ytr[j], device=dev)
    if step < WARMUP:
        EMB.eval()
        with torch.no_grad():
            e = embed_train(wav)                      # projections and class weights warm up on the frozen encoder
    else:
        EMB.train(); e = embed_train(wav)
    pv = proj_v(e); pf = proj_f(CLIP_T[torch.tensor(rws, device=dev)])
    loss_al = lout(pv, pf, lab); loss_sp = aam_loss(e, lab); loss = loss_al + LAM_SPK * loss_sp
    if not torch.isfinite(loss):
        n_bad += 1; opt.zero_grad()
        if n_bad >= 3 and USE_HALF[0]:
            USE_HALF[0] = False; print('non-finite loss x3 -> encoder runs in fp32 from here'); torch.cuda.empty_cache()
        assert n_bad < 30, 'training diverged (30 non-finite losses) - tell Claude'
        continue
    opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    step += 1; n_done += 1
    if n_done == 20:
        print('  speed %.2f s/step -> remaining %d steps about %.0f min' % ((time.time() - t0) / 20, TOTAL - step, (TOTAL - step) * (time.time() - t0) / 20 / 60))
    if step == WARMUP:
        ref = evaluate(); note('step %5d REFERENCE frozen encoder + warmed-up projections: %s' % (step, fmt(ref))); save_ckpt()
    elif step % EVAL_EVERY == 0 or step == TOTAL:
        r = evaluate(); improved = r['val_ext'] < best - 0.05
        hist.append((step, round(r['val_ext'], 2), round(r['ch_std'], 2), round(r['ch_gender'], 2), round(float(loss_al), 3), round(float(loss_sp), 3)))
        if improved:
            best, best_step, wait = r['val_ext'], step, 0
            torch.save({'emb': EMB.state_dict(), 'proj_v': proj_v.state_dict(), 'proj_f': proj_f.state_dict(), 'step': step, 'val': r, 'ref': ref, 'cfg': CFG}, BESTP)
        else:
            wait += 1
        note('step %5d | loss align %.3f spk %.3f | %s | %.0f min%s' % (step, float(loss_al), float(loss_sp), fmt(r), (time.time() - t0) / 60, '  <- best, saved' if improved else ''))
        save_ckpt()
if ref is None and os.path.exists(BESTP):
    ref = torch.load(BESTP, map_location='cpu', weights_only=False).get('ref')
best_ck = torch.load(BESTP, map_location=dev, weights_only=False); EMB.load_state_dict(best_ck['emb']); EMB.eval()
note('fine-tuning done: best held-out external EER %.2f at step %d (reference at step %d: %.2f); challenge std %.2f -> %.2f, gender %.2f -> %.2f (projection scorer, not the final system)' % (
    best_ck['val']['val_ext'], best_ck['step'], WARMUP, ref['val_ext'] if ref else float('nan'), ref['ch_std'] if ref else float('nan'), best_ck['val']['ch_std'], ref['ch_gender'] if ref else float('nan'), best_ck['val']['ch_gender']))


In [ ]:
# Cell 4 [NB40 v2 6-Nov] - fine-tuned ECAPA on FULL clips for every training row -> features/ecapa_ft.npz (train/mav/extra)
#          + features/vox_<tag>_ecapa_ft.npy; sanity: identity gap and how far the encoder moved. Skipped if the files exist.
#          v2: the challenge train has clips of several minutes; 16 of them in one batch + the training state still on the GPU = CUDA OOM (5 Nov).
#          Fix: free the training objects first; length-sorted batches also stop at MAX_BATCH_SEC of padded audio; any batch that still
#          runs out of memory is split in halves (down to one clip). Padding is masked inside ECAPA, so smaller batches give the same embeddings.
import gc
for _n in ['opt', 'scaler', 'ck', 'best_ck', 'e', 'pv', 'pf', 'wav', 'lab', 'loss', 'loss_al', 'loss_sp']:
    if _n in globals():
        del globals()[_n]
gc.collect(); torch.cuda.empty_cache()
EMB.load_state_dict(torch.load(BESTP, map_location=dev, weights_only=False)['emb']); EMB.eval()   # best checkpoint, whatever the session did before
MAX_BATCH_SEC = 240.0
def enc_ecapa_paths(paths, bs=16, desc='ecapa'):
    """full-clip ECAPA with whatever weights ECAPA.mods.embedding_model currently holds; memory-safe batching (NB40 v2)"""
    ECAPA.mods.embedding_model.eval(); out = np.zeros((len(paths), 192), np.float32)
    dur = np.array([sf.info(p).duration for p in paths]); order = np.argsort(dur)
    batches, cur = [], []
    for j in order:                                   # ascending length: padded cost of adding j = (len(cur) + 1) * dur[j]
        if cur and (len(cur) >= bs or (len(cur) + 1) * dur[j] > MAX_BATCH_SEC):
            batches.append(np.array(cur)); cur = []
        cur.append(j)
    if cur:
        batches.append(np.array(cur))
    def run(idx):
        try:
            b, rl = pad(paths, idx)
            with torch.no_grad():
                return ECAPA.encode_batch(b, rl).squeeze(1).float().cpu().numpy()
        except (torch.cuda.OutOfMemoryError, RuntimeError) as ex:
            if 'out of memory' not in str(ex).lower():
                raise
            torch.cuda.empty_cache()
            if len(idx) == 1:
                raise
            h = len(idx) // 2; return np.concatenate([run(idx[:h]), run(idx[h:])])
    for idx in tqdm(batches, desc=desc):
        out[idx] = run(idx)
    return out
print('clip length check: challenge train max %.0f s, median %.0f s' % (max(sf.info(p).duration for p in WAV_CH), np.median([sf.info(p).duration for p in WAV_CH])))
OUT_NPZ = os.path.join(FEATDIR, 'ecapa_ft.npz'); VOX_OUT = {t: os.path.join(FEATDIR, 'vox_%s_ecapa_ft.npy' % t) for t in TAGS}
if os.path.exists(OUT_NPZ) and all(os.path.exists(p) for p in VOX_OUT.values()):
    EFT = dict(np.load(OUT_NPZ)); print('fine-tuned features already on Drive')
else:
    t0 = time.time()
    EFT = {'train': enc_ecapa_paths(WAV_CH, desc='ecapa_ft challenge'), 'mav': enc_ecapa_paths(mav_v, desc='ecapa_ft mav'), 'extra': enc_ecapa_paths(extra_v, desc='ecapa_ft extra')}
    np.savez(OUT_NPZ, **EFT)
    for t in TAGS:
        E = enc_ecapa_paths(VOX[t]['wav'], desc='ecapa_ft ' + t); assert len(E) == len(VOX[t]['spk']); np.save(VOX_OUT[t], E)
    note('fine-tuned ECAPA extracted for all rows in %.0f min -> ecapa_ft.npz + %s' % ((time.time() - t0) / 60, ', '.join(os.path.basename(p) for p in VOX_OUT.values())))
assert all(np.isfinite(EFT[s]).all() for s in EFT), 'non-finite fine-tuned features - tell Claude'
def gap(E, S, n=600, seed=0):
    r = np.random.default_rng(seed); idx = r.choice(len(S), min(n, len(S)), replace=False); Z = l2(E[idx]); C = Z @ Z.T
    same = (S[idx][:, None] == S[idx][None, :]) & ~np.eye(len(idx), dtype=bool); diff = S[idx][:, None] != S[idx][None, :]
    return C[same].mean() - C[diff].mean()
note('identity gap (same - different cosine) on challenge rows: frozen ECAPA %.3f | fine-tuned %.3f' % (gap(Xv, y), gap(EFT['train'], y)))
note('frozen vs fine-tuned ECAPA, challenge rows: median cosine %.3f (1.0 = unchanged encoder)' % np.median((l2(Xv) * l2(EFT['train'])).sum(1)))
NOTES40 = SUMMARY


In [ ]:
# Cell 5 - features, labels, CV folds, harness (NB36 Cell 2 verbatim, same as NB39 Cell 5)
import os, re, json, time, zipfile, copy
import numpy as np, pandas as pd
import torch.nn as nn, torch.nn.functional as Fn
from sklearn.metrics import roc_curve
BASE = '/content/drive/MyDrive'
PROJECT = os.path.join(BASE, 'FLAG2027'); FEATDIR = os.path.join(PROJECT, 'features'); RESDIR = os.path.join(PROJECT, 'results')
ZIP_DIR = json.load(open(os.path.join(PROJECT, 'paths.json')))['ZIP_DIR']
SUMMARY = []
def note(msg):
    print(msg); SUMMARY.append(msg)
LISTS = '/content/flag_lists'
if not os.path.exists(LISTS + '/done'):
    with zipfile.ZipFile(os.path.join(ZIP_DIR, 'train_set.zip')) as zf:
        zf.extractall(LISTS, members=[m for m in zf.namelist() if m.endswith(('.txt', '.csv'))])
    open(LISTS + '/done', 'w').close()
TXT = [os.path.join(r, f) for r, _, fs in os.walk(LISTS) for f in fs if f.endswith(('.txt', '.csv'))]
def find_one(*parts):
    hits = [p for p in TXT if all(x in p for x in parts)]
    assert len(hits) == 1, 'looking for %s -> found %s' % (parts, hits)
    return hits[0]
def read_feat(path, dim):
    df = pd.read_csv(path, header=None)
    return df.iloc[:, :dim].values.astype(np.float32), (df.iloc[:, dim].values if df.shape[1] > dim else None)
Xv, y = read_feat(find_one('train_English_voices.csv'), 192); y = y.astype(int)
train_lines = [l.split() for l in open(find_one('train_English.txt')) if l.strip()]
cls2spk = {int(l[5]): l[4] for l in train_lines}
meta = pd.read_csv(os.path.join(ZIP_DIR, 'meta_file_train_set.csv'), header=None, names=['spk', 'g'])
spk2g = dict(zip(meta.spk.astype(str).str.strip(), meta.g.astype(str).str.strip().str.lower()))
ids = np.unique(y); id_gender = {c: spk2g[cls2spk[c]] for c in ids}
MAV = dict(np.load(os.path.join(FEATDIR, 'mavceleb_v1v3.npz'))); EXTRA = dict(np.load(os.path.join(FEATDIR, 'mavceleb_extra.npz')))
ML = np.load(os.path.join(FEATDIR, 'mav_lang.npz')); EXT_SPK, EXT_LANG = ML['spk'], ML['lang']
assert list(EXT_SPK) == list(MAV['spk']) + list(EXTRA['spk']), 'mav_lang.npz does not match the feature files - tell Claude'
FEAT = {'ecapa': {'train': Xv, 'mav': MAV['ecapa'], 'extra': EXTRA['ecapa']}}
for name in ['clip', 'wavlm_sv', 'r293', 'redim', 'audeer', 'whisper']:
    d = np.load(os.path.join(FEATDIR, name + '.npz')); FEAT[name] = {s: d[s] for s in d.files if not s.endswith('_ag')}
FEAT['clip']['mav'], FEAT['clip']['extra'] = MAV['clip'], EXTRA['clip']; FEAT['wavlm_sv']['mav'], FEAT['wavlm_sv']['extra'] = MAV['wavlm_sv'], EXTRA['wavlm_sv']
FEAT['r293']['extra'] = EXTRA['r293']
for name in FEAT:
    for s in ['train', 'mav', 'extra']:
        assert s in FEAT[name] and len(FEAT[name][s]) == {'train': len(y), 'mav': len(MAV['spk']), 'extra': len(EXTRA['spk'])}[s], '%s/%s missing or wrong length - tell Claude' % (name, s)
N_CH = len(y); ext_spk = list(EXT_SPK)
ext_lab = {s: 1000 + i for i, s in enumerate(sorted(set(ext_spk)))}
Y_ALL = np.r_[y, np.array([ext_lab[s] for s in ext_spk])]; D_ALL = np.r_[np.zeros(N_CH, int), np.ones(len(ext_spk), int)]; EXT = N_CH + np.arange(len(ext_spk))
def l2(X):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
def build(names, split):
    return np.concatenate([l2(FEAT[n][split]) for n in names], 1).astype(np.float32)
def build_train(names):
    return np.concatenate([build(names, 'train'), build(names, 'mav'), build(names, 'extra')])
NAP_K = 8
def nap_basis(V_ext, k):
    diffs = []
    for s in np.unique(EXT_SPK):
        m = EXT_SPK == s; langs = np.unique(EXT_LANG[m])
        if len(langs) < 2:
            continue
        mu = [V_ext[m & (EXT_LANG == l)].mean(0) for l in langs]
        for a in range(len(mu)):
            for b in range(a + 1, len(mu)):
                diffs.append(mu[a] - mu[b])
    Dm = np.stack(diffs); U = np.linalg.svd(Dm - Dm.mean(0), full_matrices=False)[2][:k]
    return U.astype(np.float32), len(diffs)
def nap_apply(X, U):
    return (X - (X @ U.T) @ U).astype(np.float32)
def eer(labels, scores):
    fpr, tpr, _ = roc_curve(labels, scores); fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr)); return (fpr[i] + fnr[i]) / 2 * 100
def make_pairs(idx, n=2000, same_gender_neg=False, seed=0):
    rng = np.random.default_rng(seed); spk_ = np.unique(y[idx]); rows_ = {s: idx[y[idx] == s] for s in spk_}
    V, F, L = [], [], []
    for _ in range(n):
        s = rng.choice(spk_); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[s])); L.append(1)
        s = rng.choice(spk_)
        cand = [t for t in spk_ if t != s and (not same_gender_neg or id_gender[t] == id_gender[s])] or [t for t in spk_ if t != s]
        t = rng.choice(cand); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[t])); L.append(0)
    return np.array(V), np.array(F), np.array(L)
K = 5; rng = np.random.default_rng(42); folds = [[] for _ in range(K)]
for gg in sorted(set(id_gender.values())):
    members = [c for c in ids if id_gender[c] == gg]; rng.shuffle(members)
    for i, c in enumerate(members):
        folds[i % K].append(c)
FOLD_DATA = []
for k in range(K):
    val_ids = set(folds[k]); va_idx = np.where([c in val_ids for c in y])[0]; tr_idx = np.where([c not in val_ids for c in y])[0]
    FOLD_DATA.append((tr_idx, [make_pairs(va_idx, seed=k), make_pairs(va_idx, same_gender_neg=True, seed=100 + k)]))
_PROTO = {}
def max_sep_prototypes(n):
    if n in _PROTO:
        return _PROTO[n]
    P = np.array([[1.0, -1.0]])
    for k in range(2, n):
        Q = np.zeros((k, k + 1)); Q[0, 0] = 1.0; Q[0, 1:] = -1.0 / k; Q[1:, 1:] = np.sqrt(1 - 1.0 / k ** 2) * P; P = Q
    _PROTO[n] = torch.tensor(P.T, dtype=torch.float32); return _PROTO[n]
class Branch(nn.Module):
    def __init__(self, d_in, d=128):
        super().__init__(); self.net = nn.Sequential(nn.Linear(d_in, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Dropout(0.5))
    def forward(self, x):
        return Fn.normalize(self.net(x), dim=1)
class FOP(nn.Module):
    def __init__(self, n_class, d_face, d_voice, d=128):
        super().__init__()
        self.face, self.voice = Branch(d_face, d), Branch(d_voice, d)
        self.att = nn.Sequential(nn.Linear(2 * d, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Linear(d, d))
        self.cls = nn.Linear(d, n_class - 1); self.register_buffer('P', max_sep_prototypes(n_class))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight); nn.init.constant_(m.bias, 0.01)
    def embed(self, xf, xv):
        f, v = self.face(xf), self.voice(xv); return torch.tanh(f), torch.tanh(v), f, v
    def forward(self, xf, xv):
        ft, vt, f, v = self.embed(xf, xv); a = torch.sigmoid(self.att(torch.cat([f, v], 1))); fused = ft * a + (1 - a) * vt
        return fused, self.cls(fused) @ self.P.t(), ft, vt
def opl_loss(feat, lab):
    feat = Fn.normalize(feat, dim=1); dot = feat @ feat.t(); same = (lab[:, None] == lab[None, :]).float()
    pos = same - torch.eye(len(lab), device=feat.device); neg = 1 - same
    return (1 - (pos * dot).sum() / (pos.sum() + 1e-6)) + 0.7 * (neg * dot.abs()).sum() / (neg.sum() + 1e-6)
def nce_loss(ft, vt, lab, tau):
    """symmetric identity-aware InfoNCE between the two scored embeddings: every same-identity row counts as a positive"""
    S = Fn.normalize(ft, dim=1) @ Fn.normalize(vt, dim=1).t() / tau; same = lab[:, None] == lab[None, :]
    def one(S_):
        logp = S_ - torch.logsumexp(S_, 1, keepdim=True); return -torch.logsumexp(logp.masked_fill(~same, -1e4), 1).mean()
    return 0.5 * (one(S) + one(S.t()))
@torch.no_grad()
def same_score(model, xf, xv):
    model.eval(); mf, sf_, mv, sv = model.norm; ft, vt, _, _ = model.embed((xf - mf) / sf_, (xv - mv) / sv)
    return -(ft - vt).norm(dim=1).cpu().numpy()
def dnorm(F_raw, V_raw, tr_idx):
    F, V = F_raw.copy(), V_raw.copy(); ch = None
    for d in np.unique(D_ALL[tr_idx]):
        trd = tr_idx[D_ALL[tr_idx] == d]; alld = np.where(D_ALL == d)[0]
        st = (F_raw[trd].mean(0), F_raw[trd].std(0) + 1e-6, V_raw[trd].mean(0), V_raw[trd].std(0) + 1e-6)
        F[alld] = (F_raw[alld] - st[0]) / st[1]; V[alld] = (V_raw[alld] - st[2]) / st[3]
        if d == 0:
            ch = st
    return F, V, ch
SEEDS = [1, 2, 3]; P_ID, K_CLIP, ALPHA, LR, MAX_EP, PATIENCE, STEPS = 16, 8, 2.0, 1e-4, 100, 15, 60
def train_head(F_raw, V_raw, all_tr, vsets=None, seed=1, cfg=None, max_epochs=MAX_EP):
    """cfg keys: align in {None,'mse','nce'}, w (alignment weight), tau, repair (bool), p_in (input dropout)"""
    cfg = cfg or {}; align, w_al, tau, repair, p_in = cfg.get('align'), cfg.get('w', 1.0), cfg.get('tau', 0.07), cfg.get('repair', False), cfg.get('p_in', 0.0)
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    F, V, ch = dnorm(F_raw, V_raw, all_tr); XF = torch.tensor(F, device=dev); XV = torch.tensor(V, device=dev)
    cl = np.unique(Y_ALL[all_tr]); rm = {c: i for i, c in enumerate(cl)}
    ytr_np = np.array([rm[c] for c in Y_ALL[all_tr]]); ytr = torch.tensor(ytr_np, device=dev); pos_by_cls = [np.where(ytr_np == c)[0] for c in range(len(cl))]
    tr = torch.tensor(all_tr, device=dev); model = FOP(len(cl), XF.shape[1], XV.shape[1]).to(dev)
    model.norm = (XF[tr].mean(0), XF[tr].std(0) + 1e-6, XV[tr].mean(0), XV[tr].std(0) + 1e-6); mf, sf_, mv, sv = model.norm
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=0.01); best, best_ep, best_state, wait = 1e9, max_epochs, None, 0
    for ep in range(1, max_epochs + 1):
        model.train()
        for _ in range(STEPS):
            chosen = rng.choice(len(cl), P_ID, replace=False)
            j_np = np.concatenate([rng.choice(pos_by_cls[c], K_CLIP, replace=len(pos_by_cls[c]) < K_CLIP) for c in chosen])
            jv_np = np.concatenate([g[rng.permutation(K_CLIP)] for g in j_np.reshape(P_ID, K_CLIP)]) if repair else j_np   # voice from another clip of the same identity
            j, jv = torch.tensor(j_np, device=dev), torch.tensor(jv_np, device=dev)
            xf = (XF[tr[j]] - mf) / sf_; xv = (XV[tr[jv]] - mv) / sv
            if p_in > 0:
                xf = Fn.dropout(xf, p_in, True); xv = Fn.dropout(xv, p_in, True)
            fused, logits, ft, vt = model(xf, xv); lab = ytr[j]
            loss = Fn.cross_entropy(logits, lab) + ALPHA * opl_loss(fused, lab)
            if align == 'mse':
                loss = loss + w_al * ((ft - vt) ** 2).mean()
            elif align == 'nce':
                loss = loss + w_al * nce_loss(ft, vt, lab, tau)
            opt.zero_grad(); loss.backward(); opt.step()
        if vsets is None:
            continue
        score = np.mean([eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets])
        if score < best - 0.05:
            best, best_ep, best_state, wait = score, ep, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_ep, XF, XV, ch
R4 = ['ecapa', 'wavlm_sv', 'r293', 'redim', 'audeer']; VOICE = R4 + ['whisper']
F_RAW = build_train(['clip']); V_full = build_train(VOICE); U_NAP, n_d = nap_basis(V_full[N_CH:], NAP_K); V_RAW = nap_apply(V_full, U_NAP)
note('harness ready: %d challenge rows, %d MAV rows, face %d-d, voice %d-d after NAP-%d (%d language-difference vectors)' % (N_CH, len(EXT), F_RAW.shape[1], V_RAW.shape[1], NAP_K, n_d))

In [ ]:
# Cell 6 [NB40 v1 5-Nov] - CV: the fine-tuned ECAPA block in system E's recipe (face CLIP | voice R4 + whisper, NAP-8; CE + 2*OPL + 0.5*L_out tau 0.07).
#          V_eft_add = E's voice stack + ecapa_ft; V_eft_rep = ecapa_ft in place of the frozen ecapa. R0_ref copied from nb39_cv.csv.
#          Stage A usual folds seeds 1-3; stage B NEW folds + seeds 4-6 (the only stage that counts). Training-side only; per-trial scoring.
SUMMARY = list(NOTES40) + SUMMARY
d = np.load(os.path.join(FEATDIR, 'ecapa_ft.npz')); FEAT['ecapa_ft'] = {s: d[s] for s in ['train', 'mav', 'extra']}
assert all(len(FEAT['ecapa_ft'][s]) == len(FEAT['clip'][s]) for s in ['train', 'mav', 'extra']), 'ecapa_ft rows do not match - tell Claude'
def lout_loss(ft, vt, lab, tau):
    S = Fn.normalize(ft, dim=1) @ Fn.normalize(vt, dim=1).t() / tau; P = (lab[:, None] == lab[None, :]).float()
    def one(M):
        return -(P * Fn.log_softmax(M, 1)).sum(1).div(P.sum(1)).mean()
    return 0.5 * (one(S) + one(S.t()))
def train_e(F_raw, V_raw, all_tr, vsets, seed, w_al=0.5, tau=0.07):
    """system E's recipe (= NB35 train_lout = NB39 train_e): 16 ids x 8 clips, CE + 2*OPL + 0.5*L_out, early stopping on the fold's pairs"""
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    F, V, ch = dnorm(F_raw, V_raw, all_tr); XF = torch.tensor(F, device=dev); XV = torch.tensor(V, device=dev)
    cl = np.unique(Y_ALL[all_tr]); rm = {c: i for i, c in enumerate(cl)}
    ytr_np = np.array([rm[c] for c in Y_ALL[all_tr]]); ytr = torch.tensor(ytr_np, device=dev); pos_by_cls = [np.where(ytr_np == c)[0] for c in range(len(cl))]
    tr = torch.tensor(all_tr, device=dev); model = FOP(len(cl), XF.shape[1], XV.shape[1]).to(dev)
    model.norm = (XF[tr].mean(0), XF[tr].std(0) + 1e-6, XV[tr].mean(0), XV[tr].std(0) + 1e-6); mf, sf_, mv, sv = model.norm
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=0.01); best, best_ep, best_state, wait = 1e9, MAX_EP, None, 0
    for ep in range(1, MAX_EP + 1):
        model.train()
        for _ in range(STEPS):
            chosen = rng.choice(len(cl), P_ID, replace=False)
            j = torch.tensor(np.concatenate([rng.choice(pos_by_cls[c], K_CLIP, replace=len(pos_by_cls[c]) < K_CLIP) for c in chosen]), device=dev)
            fused, logits, ft, vt = model((XF[tr[j]] - mf) / sf_, (XV[tr[j]] - mv) / sv); lab = ytr[j]
            loss = Fn.cross_entropy(logits, lab) + ALPHA * opl_loss(fused, lab) + w_al * lout_loss(ft, vt, lab, tau)
            opt.zero_grad(); loss.backward(); opt.step()
        score = np.mean([eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets])
        if score < best - 0.05:
            best, best_ep, best_state, wait = score, ep, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    model.load_state_dict(best_state); return model, best_ep, XF, XV
COMBOS = {'V_eft_add': (['clip'], VOICE + ['ecapa_ft']), 'V_eft_rep': (['clip'], ['ecapa_ft'] + [v for v in VOICE if v != 'ecapa'])}
def inputs(fn_, vn_):
    F_raw = build_train(fn_); V_full = build_train(vn_); U, _ = nap_basis(V_full[N_CH:], NAP_K); return F_raw, nap_apply(V_full, U)
rngB = np.random.default_rng(7); foldsB = [[] for _ in range(K)]
for gg in sorted(set(id_gender.values())):
    members = [c for c in ids if id_gender[c] == gg]; rngB.shuffle(members)
    for i, c in enumerate(members):
        foldsB[i % K].append(c)
FOLD_B = []
for k in range(K):
    val_ids = set(foldsB[k]); va_idx = np.where([c in val_ids for c in y])[0]; tr_idx = np.where([c not in val_ids for c in y])[0]
    FOLD_B.append((tr_idx, [make_pairs(va_idx, seed=200 + k), make_pairs(va_idx, same_gender_neg=True, seed=300 + k)]))
CSV40 = os.path.join(RESDIR, 'nb40_cv.csv')
if not os.path.exists(CSV40):
    r = pd.read_csv(os.path.join(RESDIR, 'nb39_cv.csv')); r = r[r.combo == 'R0_ref'][['stage', 'combo', 'seed', 'fold', 'best_ep', 'std', 'gender']].copy()
    assert len(r) == 30, 'R0_ref incomplete in nb39_cv.csv - tell Claude'
    r.to_csv(CSV40, index=False); print('R0_ref copied from NB39 (30 runs)')
done40 = pd.read_csv(CSV40); t0 = time.time()
def run40(stage, names, seeds, fold_data):
    global done40
    for name in names:
        todo = [(seed, k) for seed in seeds for k in range(K) if not ((done40.stage == stage) & (done40.combo == name) & (done40.seed == seed) & (done40.fold == k)).any()]
        if not todo:
            continue
        F_raw, V_raw = inputs(*COMBOS[name])
        for seed in seeds:
            for k in range(K):
                if (seed, k) not in todo:
                    continue
                tr_idx, vsets = fold_data[k]
                try:
                    model, best_ep, XF, XV = train_e(F_raw, V_raw, np.concatenate([tr_idx, EXT]), vsets, seed)
                    e_std, e_gen = [eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets]
                except Exception as e:
                    print(name, 'FAILED', repr(e)[:300]); best_ep, e_std, e_gen = -1, 50.0, 50.0
                row = pd.DataFrame([dict(stage=stage, combo=name, seed=seed, fold=k, best_ep=best_ep, std=e_std, gender=e_gen)])
                row.to_csv(CSV40, mode='a', header=False, index=False); done40 = pd.concat([done40, row])
            print('%s %-10s seed %d done | %.0f min' % (stage, name, seed, (time.time() - t0) / 60))
        del F_raw, V_raw
def tab40(stage):
    R = pd.read_csv(CSV40); R = R[R.stage == stage].copy(); R['overall'] = (R['std'] + R['gender']) / 2
    return R.groupby(['combo', 'seed'])[['std', 'gender', 'overall', 'best_ep']].mean().groupby('combo').mean().round(2).sort_values('overall')
def adopt(TB, c):
    refB = TB.loc['R0_ref']; dB, dg, ds = TB.loc[c, 'overall'] - refB['overall'], TB.loc[c, 'gender'] - refB['gender'], TB.loc[c, 'std'] - refB['std']
    return (dB <= -0.4 or (dg <= -0.5 and ds <= 0.2)), dB, ds, dg
run40('A', list(COMBOS), [1, 2, 3], FOLD_DATA)
run40('B', list(COMBOS), [4, 5, 6], FOLD_B)
TA, TB = tab40('A'), tab40('B')
note('NB40 STAGE A (usual folds, seeds 1-3; R0_ref = system E recipe):'); note(TA.to_string())
note('NB40 STAGE B (NEW folds + NEW seeds 4-6):'); note(TB.to_string())
note('VERDICTS (stage B only: overall <= -0.4, or gender <= -0.5 with std <= +0.2 vs R0_ref):')
WIN = []
for c in COMBOS:
    ok, dB, ds, dg = adopt(TB, c)
    note('%-10s %s: A %+.2f | B %+.2f (std %+.2f, gender %+.2f) -> %s' % (c, 'fine-tuned ECAPA ' + ('added' if c.endswith('add') else 'replacing frozen'), TA.loc[c, 'overall'] - TA.loc['R0_ref', 'overall'], dB, ds, dg, 'ADOPT' if ok else 'reject'))
    if ok:
        WIN.append(c)
if WIN:
    note('NEXT: ADOPTED (%s). The VoxCeleb files vox_<tag>_ecapa_ft.npy and the encoder final/ecapa_ft.pt are already on Drive -> Claude builds NB20 v10 with system H (dev upload, then eval).' % ', '.join(WIN))
else:
    note('NEXT: nothing adopted; system E stays; the dry run and the eval submission proceed as planned.')
print('=' * 70); print('SUMMARY  (copy everything between the lines and paste to Claude)'); print('=' * 70)
for s in SUMMARY:
    print(s)
print('=' * 70)
